In [ ]:
import os
import sys
import numpy as np
import torch
import torch.nn as nn
from torch.autograd import Variable
from torch.utils.data import DataLoader
from torchvision.transforms import Compose
from tqdm import tqdm
import mlflow.pytorch
import imgaug.augmenters as iaa
from pprint import pprint
import pandas as pd
import cv2

import mlflow

import matplotlib.pyplot as plt
%matplotlib inline


sys.path.append('../../')

from src.data import TrainValTestSplitter, MURASubset
from src.data.transforms import GrayScale, Padding, Resize, HistEqualisation, MinMaxNormalization, ToTensor
from src.features.augmentation import Augmentation
from src.models import BottleneckAutoencoder
from src.models.torchsummary import summary
from src import DATA_PATH, MODELS_DIR, MLFLOW_TRACKING_URI

## Helper function to get image in right format

In [ ]:
size_sample = 20 # set how many image to be analysed

In [ ]:
#MLFlow server must be running at port 5001, but default is 5000

client = mlflow.tracking.MlflowClient(MLFLOW_TRACKING_URI)
client.list_experiments()

In [ ]:
run_id = '850f48763e2745f282afb942465b71e5'
experiment = client.get_experiment('1')
path = f'{experiment.artifact_location}/{run_id}/artifacts/baseline_autoencoder_aug/data/model.pth'
path
path_csv = f'{experiment.artifact_location}/{run_id}/artifacts/baseline_autoencoder_aug/data/evaluate_aug_1.csv'
path_loss = f'{experiment.artifact_location}/{run_id}/artifacts/baseline_autoencoder_aug/data/losses_aug_1_2.csv'

# 1. Augmentation: 5 different augmentation, 1000 Epoch

## 1.1 Get model

In [ ]:
# get data path
data_path = f'{DATA_PATH}/XR_HAND_CROPPED'
# get data split
splitter = TrainValTestSplitter(data_path) 
# set up parameters
run_params = {
    'batch_size': 32,
    'image_resolution': (512, 512),
    'num_epochs': 500,
    'batch_normalisation': True,
    'pipeline': {
        'hist_equalisation': True,
        'cropped': False,
    }
}
# set transformation steps
composed_transforms = Compose([GrayScale(),
                               Padding(),
                               Resize(run_params['image_resolution']),
                               HistEqualisation(active=run_params['pipeline']['hist_equalisation']),
                               MinMaxNormalization(),
                               ToTensor()])

# set cpu
num_workers = 6

# get data loadder
validation = MURASubset(filenames=splitter.data_val.path, true_labels=splitter.data_val.label,
                        patients=splitter.data_val.patient, transform=composed_transforms)
val_loader = DataLoader(validation, batch_size=run_params['batch_size'], shuffle=True, num_workers=num_workers)

# set device, change if needed
device = "cuda"
# load model from path
model = torch.load(path, map_location=lambda storage, loc: storage)
# define loss
outer_loss = nn.MSELoss(reduction='none')
# set model to evaluation mode 
model.eval().to(device)

In [ ]:
# composed_transforms = Compose([GrayScale(),
#                                Padding(max_shape=(750, 750)),  # max_shape - max size of image after augmentation
#                                Resize(run_params['image_resolution']),
#                                HistEqualisation(active=run_params['pipeline']['hist_equalisation']),
#                                MinMaxNormalization(),
#                               ToTensor()])

## Dataset loaders
# print(f'\nDATA SPLIT:')
# splitter = TrainValTestSplitter(path_to_data=data_path)
# validation = MURASubset(filenames=splitter.data_val.path, true_labels=splitter.data_val.label,
#                         patients=splitter.data_val.patient, transform=composed_transforms)

# val_loader = DataLoader(validation, batch_size=run_params['batch_size'], shuffle=True, num_workers=num_workers)

## 1.2 Get data frame with losses and images from the last validation

In [ ]:
augmentation_df_1 = pd.read_csv(path_csv)
print(augmentation_df_1)

### 1.2.1 False Negative

In [ ]:
false_negative = augmentation_df_1[augmentation_df_1['True label'] == 1].nsmallest(size_sample, 'Loss')
false_negative.style.set_properties(subset=['File name'], **{'width': '300px'})

In [ ]:
#files_false_negative_path = [x[3:] for x in false_negative['File name'] ]
files_false_negative_path = false_negative['File name']
files_false_negative = MURASubset(filenames=files_false_negative_path, transform=composed_transforms)

In [ ]:
fig, ax = plt.subplots(nrows=size_sample, ncols=2, figsize=(20, 8*size_sample))

for i in range(size_sample):
    image_inp = files_false_negative[i]['image'][None,:,:,:]
    inp = image_inp.to(device)
    output = model(inp)
    
    ax[i, 0].imshow(image_inp.numpy()[0, 0, :, :], cmap='gray', vmin=0, vmax=1)
    ax[i, 1].imshow(output.cpu().detach().numpy()[0, 0, :, :], cmap='gray', vmin=0, vmax=1)

### 1.2.2 True Negative

In [ ]:
true_negative = augmentation_df_1[augmentation_df_1['True label'] == 0].nsmallest(size_sample, 'Loss')
true_negative.style.set_properties(subset=['File name'], **{'width': '300px'})


In [ ]:
#files_true_negative_path = [x[3:] for x in true_negative['File name'] ]
files_true_negative_path = true_negative['File name'] 
files_true_negative = MURASubset(filenames=files_true_negative_path, transform=composed_transforms)

In [ ]:
fig, ax = plt.subplots(nrows=size_sample, ncols=2, figsize=(20, 8*size_sample))

for i in range(size_sample):
    image_inp = files_true_negative[i]['image'][None,:,:,:]
    inp = image_inp.to(device)
    output = model(inp)
    
    ax[i, 0].imshow(image_inp.numpy()[0, 0, :, :], cmap='gray', vmin=0, vmax=1)
    ax[i, 1].imshow(output.cpu().detach().numpy()[0, 0, :, :], cmap='gray', vmin=0, vmax=1)

### 1.2.3 False Positive

In [ ]:
false_positive = augmentation_df_1[augmentation_df_1['True label'] == 0].nlargest(size_sample, 'Loss')
false_positive.style.set_properties(subset=['File name'], **{'width': '300px'})

In [ ]:
#files_false_positive_path = [x[3:] for x in false_positive['File name'] ]
files_false_positive_path = false_positive['File name']
files_false_positive = MURASubset(filenames=files_false_positive_path, transform=composed_transforms)

In [ ]:
fig, ax = plt.subplots(nrows=size_sample, ncols=2, figsize=(20, 8*size_sample))

for i in range(size_sample):
    image_inp = files_false_positive[i]['image'][None,:,:,:]
    inp = image_inp.to(device)
    output = model(inp)
    
    ax[i, 0].imshow(image_inp.numpy()[0, 0, :, :], cmap='gray', vmin=0, vmax=1)
    ax[i, 1].imshow(output.cpu().detach().numpy()[0, 0, :, :], cmap='gray', vmin=0, vmax=1)

The images are missclassified because of:
 - black image without any hand
 - a lot of noise around the hand
 - complex labels that can not be reconstructed
 - some images have more rectangels around the hand
 - wired view on hand
 - hand in motion
 - brighness

### 1.2.4 True Positive

In [ ]:
true_positive = augmentation_df_1[augmentation_df_1['True label'] == 1].nlargest(size_sample, 'Loss')
true_positive.style.set_properties(subset=['File name'], **{'width': '300px'})

In [ ]:
#files_true_positive_path = [x[3:] for x in true_positive['File name'] ]
files_true_positive_path = true_positive['File name']
files_true_positive = MURASubset(filenames=files_true_positive_path, transform=composed_transforms)

In [ ]:
fig, ax = plt.subplots(nrows=size_sample, ncols=2, figsize=(20, 8*size_sample))

for i in range(size_sample):
    image_inp = files_true_positive[i]['image'][None,:,:,:]
    inp = image_inp.to(device)
    output = model(inp)
    
    ax[i, 0].imshow(image_inp.numpy()[0, 0, :, :], cmap='gray', vmin=0, vmax=1)
    ax[i, 1].imshow(output.cpu().detach().numpy()[0, 0, :, :], cmap='gray', vmin=0, vmax=1)

## 1.3 Loss per Epoch

In [ ]:
augmentation_df_loss_1 = pd.read_csv(path_loss)
print(augmentation_df_loss_1)

In [ ]:
fig = augmentation_df_loss_1.plot.line(x = 'Unnamed: 0',
                                       y = ['train', 'validation'])
fig.set_xlabel('Epoch', fontsize=18)
fig.set_ylabel('Loss', fontsize=18)
fig.set_title('Loss Augmentation: Flip', fontsize=20)

The model seems not to overfit at 1000 epochs. It looks like the loss is still decreasing with some outliers. Maybe modify learning rate.